# 03 · Utilidad del modelo, inferencia de membresía y sesgo
**Tarea no clínica:** predecir si una sesión alcanza 75 minutos usando frecuencia semanal y experiencia. Fuente: Gym, separada de Fitbit. Este modelo educativo no asigna entrenamientos ni toma decisiones sobre personas. No se despliega.

Una fila por perfil del archivo, eliminando duplicados exactos. Como no hay ID longitudinal verificable, aquí solo se afirma protección **por fila de entrenamiento**, no por persona en toda posible base. El conjunto de prueba se mantiene fuera del entrenamiento.

**Referencia:** clasificador por tabla de frecuencias de 21 combinaciones × 2 clases. **Versión privada:** el mismo clasificador, luego de perturbar el histograma con Laplace. Cada fila modifica una sola celda en una unidad, de modo que la sensibilidad L1 del vector es 1. Tras aplicar ruido independiente de escala 1/ε, recortar valores negativos y suavizar para estimar probabilidades es posprocesamiento. La comparación aísla el efecto del ruido, sin cambiar de arquitectura.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'backend/privacy.py').is_file())
sys.path.insert(0, str(ROOT))
RAW = ROOT / 'data/raw'
from backend.experiment import model_lab
results=model_lab(RAW/'gym_members_exercise_tracking.csv',repeats=30)
print("Entrenamiento / prueba:", results['train'],results['test'])
print("Sin ruido:", results['baseline'])
for v in results['private']: print(v)
print("Exactitud por grupo en prueba (referencia):",results['baseline_groups'])
print("Con ε = 1:",results['private_group_accuracy']['1.0'])

In [ ]:
import matplotlib.pyplot as plt
fig, axs=plt.subplots(1,2,figsize=(8.5,3))
for ax,metric,title in zip(axs,['balanced_accuracy','attack_auc'],['Utilidad: exactitud balanceada','Ataque: AUC de membresía']):
    ax.plot([r['epsilon'] for r in results['private']],[r[metric] for r in results['private']],marker='o',label='Con protección')
    ax.axhline(results['baseline'][metric],ls='--',color='#81724a',label='Referencia')
    ax.set(xlabel='Épsilon de entrenamiento',title=title,ylim=(0,1));ax.legend(fontsize=8)
fig.tight_layout();plt.show()

## Qué mide el ataque
Siguiendo el laboratorio sobre inferencia de membresía, se utiliza la confianza asignada a la clase correcta (equivalente en orden a la pérdida logarítmica negativa) para distinguir filas de entrenamiento y prueba. Se reporta AUC y ventaja exploratoria máxima TPR−FPR al barrer umbrales. No se trata de DP-SGD: se adapta la **evaluación del ataque** al clasificador tabular. La ventaja se estima en esta partición y puede ser optimista. Una AUC cercana a 0,5 no demuestra anonimato. La garantía formal proviene del mecanismo, su sensibilidad y la unidad protegida.

## Reflexión ética aplicada
- **Sesgo:** comparar exactitud por género registrado y tamaños de grupos; una diferencia no identifica por sí sola discriminación ni causalidad. Las etiquetas representan duración, no salud ni esfuerzo. La variación entre 30 repeticiones no incorpora la incertidumbre de otras muestras.
- **Transparencia:** separar las fuentes y mostrar finalidad, ventana, límites y ruido. Un entrenador no debe usar un promedio protegido para juzgar a una persona. Si el indicador no es fiable, omitir la cifra y explicarlo.
- **Consentimiento:** participar en estadísticas del programa es opcional y revocable para futuras publicaciones. No inventamos consentimientos para el archivo académico Fitbit. Su carácter público no autoriza cualquier uso posterior.
- **Decisión:** conservar resultados solo en laboratorio, solicitar revisión humana si se propone un uso sobre clientes y no inferir atributos sensibles de una combinación aleatoria de archivos.
- **Alcance:** estas métricas de evaluación son diagnósticos internos sobre archivos públicos, no salidas DP sobre un conjunto privado de validación. No se publican modelos ni predicciones individuales desde la API.
    
Referencias: S12 Ataques, laboratorio de membresía con DP-SGD; S13 Ética, lámina 20 (representatividad, métricas por subgrupo y revisión humana); [Dwork y Roth, mecanismos y composición](https://www.cis.upenn.edu/~aaroth/Papers/privacybook.pdf).